# Zolai Qwen3-4B QLoRA Fine-Tuning

Fine-tune [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) on Tedim Zolai language data using QLoRA on a Kaggle T4 GPU.

**What this notebook does:**
1. Installs Unsloth (2-5x faster QLoRA)
2. Loads Qwen3-4B-Base with 4-bit quantization
3. Applies LoRA adapters (r=16, alpha=32)
4. Trains on Zolai-English translation + vocabulary + grammar data
5. Saves LoRA adapter to `/kaggle/working/zolai-qwen3-4b-lora`

**Requirements:** Kaggle with GPU (T4 recommended, 15GB VRAM).

**Output:** LoRA adapter weights (~20MB) ready for merging and GGUF export.

**Dataset:** Upload `train.jsonl` (produced by `format_training_data.py`) to a Kaggle Dataset, then load it in section 5 below.

## 1. Install Dependencies

We use [Unsloth](https://github.com/unslothai/unsloth) for 2-5x faster QLoRA training with 4-bit quantization.
The install takes ~2 minutes on Kaggle.

In [ ]:
%%capture
!pip install unsloth
# If Unsloth install fails, try:
# !pip install --no-deps trl peft accelerate bitsandbytes

%%capture
!pip install unsloth
# If Unsloth install fails, try:
# !pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
%%capture
!pip install unsloth
# If Unsloth install fails, try:
# !pip install --no-deps trl peft accelerate bitsandbytes

## 2. Import Libraries & Verify GPU

Checks that a GPU is available. If not, go to **Runtime > Change runtime type > GPU**.

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found! Go to Runtime -> Change runtime type -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print("Imports OK")

import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found! Go to Runtime -> Change runtime type -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print("Imports OK")

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found! Go to Runtime -> Change runtime type -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print("Imports OK")

## 3. Load Base Model with 4-bit Quantization

Loads Qwen3-4B-Base in 4-bit (NF4) to fit on a T4 GPU (~15GB VRAM).
Unsloth handles quantization automatically via BitsAndBytes.

- `unsloth/Qwen3-4B-Base` is pre-patched for Unsloth compatibility
- `MAX_SEQ_LENGTH = 512` is sufficient for Zolai sentences (avg ~15 words)

In [ ]:
# Model configuration
BASE_MODEL = "unsloth/Qwen3-4B-Base"  # pre-patched for Unsloth
# Fallback: "Qwen/Qwen3-4B" (will be quantized on-the-fly)

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Vocab size: {len(tokenizer)}")

# Model configuration
BASE_MODEL = "unsloth/Qwen3-4B-Base"  # pre-patched for Unsloth
# Fallback: "Qwen/Qwen3-4B" (will be quantized on-the-fly)

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Vocab size: {len(tokenizer)}")

In [ ]:
# Model configuration
BASE_MODEL = "unsloth/Qwen3-4B-Base"  # pre-patched for Unsloth
# Fallback: "Qwen/Qwen3-4B" (will be quantized on-the-fly)

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Vocab size: {len(tokenizer)}")

## 4. Apply LoRA Adapters

QLoRA adds small trainable adapter layers while keeping the base model frozen.

- **r=16**: adapter rank (balance of capacity vs memory)
- **alpha=32**: scaling factor (2x rank is standard)
- **dropout=0.05**: regularization to prevent overfitting
- **target_modules**: all linear layers for maximum adaptation
- **gradient_checkpointing**: 60% less VRAM with Unsloth

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 5. Load Training Data

Upload your `train.jsonl` to a Kaggle Dataset (created by `format_training_data.py`),
then load it in the cell below.

**How to upload:**
1. Go to Kaggle > Datasets > New Dataset
2. Upload `train.jsonl` (and optionally `val.jsonl`)
3. Set `DATASET_SLUG` to `your-username/zolai-training-data`
4. Make it a Code dataset so the notebook can access it

Three loading options are provided - uncomment the one that matches your setup.

In [ ]:
# Option A: Load from Kaggle Dataset (RECOMMENDED)
# Upload train.jsonl to a Kaggle Dataset, then:
DATASET_SLUG = "your-username/zolai-training-data"
dataset = load_dataset("json", data_files={
    "train": f"/kaggle/input/{DATASET_SLUG}/train.jsonl"
}, split="train")

# Option B: Load from HuggingFace Hub
# dataset = load_dataset("Zolai-AI/zolai-training-data", split="train")

# Option C: Load from local file
# dataset = load_dataset("json", data_files="/kaggle/input/zolai-train/train.jsonl", split="train")

# Option D: Demo data (for testing the pipeline)
# Uncomment the lines below to use synthetic examples instead of real data:
# import json
# from datasets import Dataset
# demo = [
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to Zolai: God created the heavens and the earth."},
#         {"role": "assistant", "content": "Pasian a vantung leh lebung piangsak hi."}
#     ]},
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to English: Pasian a topa si."},
#         {"role": "assistant", "content": "God is the Lord."}
#     ]},
# ]
# with open("/tmp/train_demo.jsonl", "w") as f:
#     for row in demo:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")
# dataset = load_dataset("json", data_files="/tmp/train_demo.jsonl", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Sample user: {dataset[0]['messages'][1]['content'][:80]}")
print(f"Sample assistant: {dataset[0]['messages'][2]['content'][:80]}")

# Option A: Load from Kaggle Dataset (RECOMMENDED)
# Upload train.jsonl to a Kaggle Dataset, then:
DATASET_SLUG = "your-username/zolai-training-data"
dataset = load_dataset("json", data_files={
    "train": f"/kaggle/input/{DATASET_SLUG}/train.jsonl"
}, split="train")

# Option B: Load from HuggingFace Hub
# dataset = load_dataset("Zolai-AI/zolai-training-data", split="train")

# Option C: Load from local file
# dataset = load_dataset("json", data_files="/kaggle/input/zolai-train/train.jsonl", split="train")

# Option D: Demo data (for testing the pipeline)
# Uncomment the lines below to use synthetic examples instead of real data:
# import json
# from datasets import Dataset
# demo = [
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to Zolai: God created the heavens and the earth."},
#         {"role": "assistant", "content": "Pasian a vantung leh lebung piangsak hi."}
#     ]},
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to English: Pasian a topa si."},
#         {"role": "assistant", "content": "God is the Lord."}
#     ]},
# ]
# with open("/tmp/train_demo.jsonl", "w") as f:
#     for row in demo:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")
# dataset = load_dataset("json", data_files="/tmp/train_demo.jsonl", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Sample user: {dataset[0]['messages'][1]['content'][:80]}")
print(f"Sample assistant: {dataset[0]['messages'][2]['content'][:80]}")

In [ ]:
# Option A: Load from Kaggle Dataset (RECOMMENDED)
# Upload train.jsonl to a Kaggle Dataset, then:
DATASET_SLUG = "your-username/zolai-training-data"
dataset = load_dataset("json", data_files={
    "train": f"/kaggle/input/{DATASET_SLUG}/train.jsonl"
}, split="train")

# Option B: Load from HuggingFace Hub
# dataset = load_dataset("Zolai-AI/zolai-training-data", split="train")

# Option C: Load from local file
# dataset = load_dataset("json", data_files="/kaggle/input/zolai-train/train.jsonl", split="train")

# Option D: Demo data (for testing the pipeline)
# Uncomment the lines below to use synthetic examples instead of real data:
# import json
# from datasets import Dataset
# demo = [
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to Zolai: God created the heavens and the earth."},
#         {"role": "assistant", "content": "Pasian a vantung leh lebung piangsak hi."}
#     ]},
#     {"messages": [
#         {"role": "system", "content": "You are a Tedim Zolai language expert."},
#         {"role": "user", "content": "Translate to English: Pasian a topa si."},
#         {"role": "assistant", "content": "God is the Lord."}
#     ]},
# ]
# with open("/tmp/train_demo.jsonl", "w") as f:
#     for row in demo:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")
# dataset = load_dataset("json", data_files="/tmp/train_demo.jsonl", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Sample user: {dataset[0]['messages'][1]['content'][:80]}")
print(f"Sample assistant: {dataset[0]['messages'][2]['content'][:80]}")

## 6. Configure Training

QLoRA hyperparameters optimized for T4 GPU (15GB VRAM):

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Epochs | 3 | Enough for style transfer without overfitting |
| Batch size | 4 | Max that fits in T4 VRAM with grad checkpointing |
| Grad accum | 4 | Effective batch = 16 (stable gradients) |
| Learning rate | 2e-4 | Standard for LoRA (higher than full fine-tune) |
| Warmup | 5% | Prevents early instability |
| Scheduler | cosine | Smooth decay to 0 |
| Optimizer | adamw_8bit | Memory-efficient AdamW |
| Precision | bf16 | Faster than fp32, sufficient precision on T4 |

In [ ]:
OUTPUT_DIR = "/kaggle/working/zolai-qwen3-4b-lora"
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = 16
LR = 2e-4
WARMUP_RATIO = 0.05

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    dataloader_num_workers=2,
)

print("Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {LR}, warmup: {WARMUP_RATIO}")
print(f"  Output: {OUTPUT_DIR}")

OUTPUT_DIR = "/kaggle/working/zolai-qwen3-4b-lora"
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = 16
LR = 2e-4
WARMUP_RATIO = 0.05

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    dataloader_num_workers=2,
)

print("Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {LR}, warmup: {WARMUP_RATIO}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
OUTPUT_DIR = "/kaggle/working/zolai-qwen3-4b-lora"
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = 16
LR = 2e-4
WARMUP_RATIO = 0.05

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    dataloader_num_workers=2,
)

print("Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {LR}, warmup: {WARMUP_RATIO}")
print(f"  Output: {OUTPUT_DIR}")

## 7. Format Data with Chat Template

Qwen3 uses `<|im_start|>` / `<im_end>` tokens. We apply the tokenizer's
chat template to each example, then mask the system+user tokens during
training so the model only learns to generate assistant responses.

In [ ]:
def format_chat(example):
    """Apply Qwen3 chat template and return {text} dict for SFTTrainer."""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)

# Show formatted example
print(dataset[0]["text"][:500])
print(f"\n... ({len(dataset[0]["text"])} chars total)")

def format_chat(example):
    """Apply Qwen3 chat template and return {text} dict for SFTTrainer."""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)

# Show formatted example
print(dataset[0]["text"][:500])
print(f"\n... ({len(dataset[0]["text"])} chars total)")

In [ ]:
def format_chat(example):
    """Apply Qwen3 chat template and return {text} dict for SFTTrainer."""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)

# Show formatted example
print(dataset[0]["text"][:500])
print(f"\n... ({len(dataset[0]["text"])} chars total)")

## 8. Train

Launch QLoRA fine-tuning with SFTTrainer.
Training should take 30-60 minutes on a T4 for ~30K examples x 3 epochs.

**Tip:** Watch the loss curve. It should decrease steadily.
If loss plateaus after epoch 1, the data might be too simple.
If loss oscillates wildly, try lowering the learning rate.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,  # pack short sentences together for efficiency
)

# Start training
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,  # pack short sentences together for efficiency
)

# Start training
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,  # pack short sentences together for efficiency
)

# Start training
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

## 9. Save LoRA Adapter

Save only the LoRA adapter weights (~20MB) instead of the full model (~8GB).
This adapter can be merged with the base model later.

In [ ]:
import os

# Save LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
print(f"Saved to {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size / 1024:.1f} KB")

# Verify adapter can be loaded
from peft import PeftModel
loaded_model = PeftModel.from_pretrained(model, OUTPUT_DIR)
print("\nAdapter verified - can be loaded successfully.")

import os

# Save LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
print(f"Saved to {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size / 1024:.1f} KB")

# Verify adapter can be loaded
from peft import PeftModel
loaded_model = PeftModel.from_pretrained(model, OUTPUT_DIR)
print("\nAdapter verified - can be loaded successfully.")

In [ ]:
import os

# Save LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
print(f"Saved to {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size / 1024:.1f} KB")

# Verify adapter can be loaded
from peft import PeftModel
loaded_model = PeftModel.from_pretrained(model, OUTPUT_DIR)
print("\nAdapter verified - can be loaded successfully.")

## 10. Test Inference

Test the fine-tuned model with a simple Zolai translation prompt.
Note: for best results, merge the adapter first (see `merge_and_export.py`).

In [ ]:
# Switch to inference mode for faster generation
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Translate to Zolai: The sun rises in the east.",
    "Translate to Zolai: I am eating rice.",
    "What does pasian mean in English?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a Tedim Zolai language expert."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    outputs = model.generate(
        inputs, max_new_tokens=128, temperature=0.7,
        top_p=0.9, do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

# Switch to inference mode for faster generation
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Translate to Zolai: The sun rises in the east.",
    "Translate to Zolai: I am eating rice.",
    "What does pasian mean in English?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a Tedim Zolai language expert."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    outputs = model.generate(
        inputs, max_new_tokens=128, temperature=0.7,
        top_p=0.9, do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

In [ ]:
# Switch to inference mode for faster generation
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Translate to Zolai: The sun rises in the east.",
    "Translate to Zolai: I am eating rice.",
    "What does pasian mean in English?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a Tedim Zolai language expert."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    outputs = model.generate(
        inputs, max_new_tokens=128, temperature=0.7,
        top_p=0.9, do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

## 11. Next Steps

After training completes:

1. **Download the adapter** from `/kaggle/working/zolai-qwen3-4b-lora`
2. **Merge and export** using `merge_and_export.py`:
   ```
   python merge_and_export.py --adapter-path ./zolai-qwen3-4b-lora --export-gguf
   ```
3. **Upload to HuggingFace** using `upload_to_hf.py`:
   ```
   python upload_to_hf.py --model-path ./zolai-qwen3-4b --repo-name Zolai-AI/zolai-qwen3-4b
   ```

**Tips for better results:**
- Add more training data (target 50K+ examples)
- Run 5-10 epochs on small datasets, 1-3 on large ones
- Include domain-specific examples (grammar exercises, vocabulary)
- Validate with native Zolai speakers